# Generate answers for the golden dataset

 When prompted, paste the current UF AI API key into the hidden input (it is not written into this notebook). The notebook uses the configured model (`gpt-oss-20b`,`llama`,`gemma` by default), reads the question set from `results/Accuracy_scores/Accuracy_scores_on_different_llm.xlsx`, and saves resumable results in `results/Accuracy_scores/`.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root. Open this notebook from the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
ACCURACY_DIR = RESULTS_DIR / "Accuracy_scores"
EVALUATION_FILE = ACCURACY_DIR / "Accuracy_scores_on_different_llm.xlsx"
if not EVALUATION_FILE.is_file():
    raise FileNotFoundError(f"Evaluation workbook not found: {EVALUATION_FILE}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation workbook: {EVALUATION_FILE.relative_to(PROJECT_ROOT)}")

Project root: /Users/rohan/Desktop/term-project-group_7
Evaluation workbook: results/Accuracy_scores/Accuracy_scores_on_different_llm.xlsx


In [9]:
import getpass
import os

entered_key = getpass.getpass(
    "Paste the current UF AI API key (hidden), or press Enter to use OPENAI_API_KEY already set: "
).strip()
api_key = entered_key or os.environ.get("OPENAI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("No API key was entered and OPENAI_API_KEY is not set in this notebook kernel.")

os.environ["OPENAI_API_KEY"] = api_key
del api_key, entered_key
print("API key is available to this notebook kernel and was not saved in the notebook.")

API key is available to this notebook kernel and was not saved in the notebook.


In [10]:
from src.retrival_pipeline import RetrievalPipeline

pipeline = RetrievalPipeline()
pipeline.initialize()
MODEL_NAME = pipeline.openai_config.model
print(f"Pipeline ready. Configured model: {MODEL_NAME}")

INITIALIZING RETRIEVAL PIPELINE
Loading embedding model: BAAI/bge-base-en-v1.5
Model loaded successfully!
 Embedding dimension: 768
✓ ChromaDB initialized at: /Users/rohan/Desktop/term-project-group_7/data/chroma_db
Retrieved existing collection: policies
  Current documents: 1887
penAI client created successfully
  Base URL: https://api.ai.it.ufl.edu
  Client type: <class 'openai.OpenAI'>
  Has chat attribute: True
  Has completions attribute: True

Pipeline initialized successfully!
Pipeline ready. Configured model: gpt-oss-20b


In [11]:
import pandas as pd

df = pd.read_excel(EVALUATION_FILE)
if "Question" not in df.columns:
    raise ValueError(f"Expected a 'Question' column in {EVALUATION_FILE.name}; found: {list(df.columns)}")
if df.empty:
    raise ValueError(f"The evaluation workbook contains no questions: {EVALUATION_FILE}")
if df["Question"].isna().any() or df["Question"].astype(str).str.strip().eq("").any():
    raise ValueError("The evaluation workbook contains a blank question. Fix it before generating answers.")

safe_model_name = "_".join(part for part in MODEL_NAME.lower().replace("-", "_").split("_") if part)
ANSWER_COLUMN = f"Generated_Answer_{MODEL_NAME}"
OUTPUT_FILE = ACCURACY_DIR / f"Golden_dataset_generated_answers_{safe_model_name}.xlsx"

# Reuse completed answers only when they belong to the same question set.
if OUTPUT_FILE.is_file():
    previous = pd.read_excel(OUTPUT_FILE)
    same_questions = (
        "Question" in previous.columns
        and previous["Question"].astype(str).tolist() == df["Question"].astype(str).tolist()
    )
    if same_questions and ANSWER_COLUMN in previous.columns:
        df[ANSWER_COLUMN] = previous[ANSWER_COLUMN]
    else:
        print("Existing output does not match this question set; starting a fresh output.")
        df[ANSWER_COLUMN] = ""
else:
    df[ANSWER_COLUMN] = ""

print(f"Questions: {len(df)}")
print(f"Answer column: {ANSWER_COLUMN}")
print(f"Output file: {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")
display(df.head())

Questions: 51
Answer column: Generated_Answer_gpt-oss-20b
Output file: results/Accuracy_scores/Golden_dataset_generated_answers_gpt_oss_20b.xlsx


,Question,Human_Answer,Gpt_Answer,Gemma_Answer,Llama_Answer,Generated_Answer_gpt-oss-20b
0,I have questions about or need help interpreti...,You may reach out to: the Responsible Office o...,If you need help interpreting the Design‑Build...,For questions about interpreting or applying t...,If your question concerns the Design‑Builder S...,
1,Where can I locate the Responsible Office or R...,The Responsible Office and Responsible Executi...,The Responsible Executive for a policy is the ...,The Responsible Executive includes the Preside...,The Responsible Executive is the senior univer...,
2,Who should I reach out to in order to schedule...,Please contact the University Policy Group to ...,You should contact the University Policy Group...,"According to the Policy on Policies document, ...","To schedule a policy consultation, you should ...",
3,How do I report a concern associated with a po...,"Any concerns related to policy, compliance or ...",To report a concern about a policy violation o...,"According to the Title IX policy, all reports ...",To report a concern about a policy violation o...,
4,Can I download or print PDF versions of the re...,The University’s regulations and policies are ...,The documents you provided do not contain any ...,The provided documents do not contain informat...,The documents you provided do not contain any ...,


In [12]:
total = len(df)
for completed, (index, row) in enumerate(df.iterrows(), start=1):
    existing_answer = df.at[index, ANSWER_COLUMN]
    if pd.notna(existing_answer) and str(existing_answer).strip():
        print(f"{completed}/{total}: already complete; skipping")
        continue

    question = str(row["Question"]).strip()
    result = pipeline.query(question=question, n_results=3)
    answer = result.get("answer")
    if not answer or not str(answer).strip():
        raise RuntimeError(f"The configured model returned an empty answer for row {index}.")

    df.at[index, ANSWER_COLUMN] = str(answer).strip()
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"{completed}/{total}: answer saved")

print(f"Completed. Results saved to: {OUTPUT_FILE}")


Querying gpt-oss-20b...
Response received successfully
1/51: answer saved

Querying gpt-oss-20b...
Response received successfully
2/51: answer saved

Querying gpt-oss-20b...
Response received successfully
3/51: answer saved

Querying gpt-oss-20b...
Response received successfully
4/51: answer saved

Querying gpt-oss-20b...
Response received successfully
5/51: answer saved

Querying gpt-oss-20b...
Response received successfully
6/51: answer saved

Querying gpt-oss-20b...
Response received successfully
7/51: answer saved

Querying gpt-oss-20b...
Response received successfully
8/51: answer saved

Querying gpt-oss-20b...
Response received successfully
9/51: answer saved

Querying gpt-oss-20b...
Response received successfully
10/51: answer saved

Querying gpt-oss-20b...
Response received successfully
11/51: answer saved

Querying gpt-oss-20b...
Response received successfully
12/51: answer saved

Querying gpt-oss-20b...
Response received successfully
13/51: answer saved

Querying gpt-oss-20b